# Generation \& Queue Review

Inspect generated CV tailoring packages and manage posting status.

In [ ]:
import sys
sys.path.insert(0, '..')
from pathlib import Path

import json
import re
import pandas as pd
from IPython.display import Markdown, display, HTML
from sqlalchemy import select
from sqlalchemy.orm import Session
from src.db import get_engine, Posting

engine = get_engine(Path('../data/jobs.db'))
pd.set_option('display.max_colwidth', 80)

## Generated postings — summary table

In [ ]:
with Session(engine) as s:
    postings = s.scalars(
        select(Posting)
        .where(Posting.status == 'generated')
        .order_by(Posting.relevance_score.desc())
    ).all()

if not postings:
    print('No generated postings yet. Run: uv run python -m src.generate')
else:
    def _verdict(content):
        if not content:
            return ''
        parts = content.split('## Verdict')
        if len(parts) < 2:
            return ''
        # first non-empty line after the header
        for line in parts[1].splitlines():
            line = line.strip()
            if line:
                return line[:100]
        return ''

    df = pd.DataFrame([
        {
            'id': p.id,
            'score': p.relevance_score,
            'level': p.level_match,
            'company': p.company,
            'title': p.title,
            'connections': len(json.loads(p.connections_json or '[]')),
            'verdict': _verdict(p.generated_content),
        }
        for p in postings
    ])
    print(f'{len(df)} generated posting(s)')
    display(df)

## Read a posting's full package

Set `POSTING_ID` to any `id` from the table above.

In [ ]:
POSTING_ID = None   # e.g. 42 — leave None to auto-pick the top-scored

with Session(engine) as s:
    if POSTING_ID is None:
        p = s.scalars(
            select(Posting)
            .where(Posting.status == 'generated')
            .order_by(Posting.relevance_score.desc())
            .limit(1)
        ).first()
    else:
        p = s.get(Posting, POSTING_ID)

if p is None:
    print('No posting found.')
else:
    conns = json.loads(p.connections_json or '[]')
    display(HTML(f"""
    <div style="background:#f8f9fa;padding:12px;border-radius:6px;margin-bottom:12px;font-family:monospace">
      <b>ID</b> {p.id} &nbsp;|&nbsp;
      <b>Score</b> {p.relevance_score}/10 &nbsp;|&nbsp;
      <b>Level</b> {p.level_match} &nbsp;|&nbsp;
      <b>Status</b> {p.status}<br>
      <b>{p.company}</b> — {p.title}<br>
      <b>Location:</b> {p.location}<br>
      <a href="{p.url}">{p.url}</a>
    </div>
    """))
    if conns:
        display(Markdown('### Network connections'))
        for c in conns:
            display(Markdown(f"- [{c['name']}]({c['linkedin_url']}) — {c['position']}"))
    display(Markdown('---'))
    display(Markdown(p.generated_content or '*No generated content.*'))

## Validate generated content structure

Checks that the skill produced all required sections.

In [ ]:
REQUIRED_SECTIONS = ['## Verdict', '## Gap analysis', '## CV edits']

with Session(engine) as s:
    gen = s.scalars(select(Posting).where(Posting.status == 'generated')).all()

issues = []
for p in gen:
    content = p.generated_content or ''
    missing = [sec for sec in REQUIRED_SECTIONS if sec not in content]
    if missing:
        issues.append({'id': p.id, 'company': p.company, 'title': p.title, 'missing': missing})

if issues:
    print(f'Warning: {len(issues)} posting(s) missing sections:')
    display(pd.DataFrame(issues))
else:
    print(f'OK: all {len(gen)} generated postings have the required sections.')

## Act on a posting

Flip status after reviewing. Valid next statuses: `reviewed`, `applied`, `skipped`.

In [ ]:
ACT_POSTING_ID = None    # e.g. 42
ACTION = None            # 'reviewed' | 'applied' | 'skipped'

ALLOWED = {'reviewed', 'applied', 'skipped'}

if ACT_POSTING_ID is not None and ACTION in ALLOWED:
    with Session(engine) as s:
        p = s.get(Posting, ACT_POSTING_ID)
        if p is None:
            print(f'Posting {ACT_POSTING_ID} not found.')
        else:
            old = p.status
            p.status = ACTION
            s.commit()
            print(f'id={ACT_POSTING_ID}: {old} -> {ACTION}')
elif ACT_POSTING_ID is not None:
    print(f'Unknown action "{ACTION}". Use one of: {ALLOWED}')
else:
    print('Set ACT_POSTING_ID and ACTION to update a posting.')

## Matched postings not yet generated

Run `uv run python -m src.generate` to process these.

In [ ]:
with Session(engine) as s:
    pending = s.scalars(
        select(Posting)
        .where(Posting.status == 'matched')
        .order_by(Posting.relevance_score.desc())
    ).all()

if not pending:
    print('Nothing pending.')
else:
    df_pending = pd.DataFrame([
        {'id': p.id, 'score': p.relevance_score, 'company': p.company, 'title': p.title,
         'connections': len(json.loads(p.connections_json or '[]'))}
        for p in pending
    ])
    print(f'{len(df_pending)} posting(s) awaiting generation:')
    display(df_pending)